In [2]:
import polars as pl

df = pl.read_ndjson("../data/omdb_raw.jsonl").unnest("raw")

# Validez: cuántos registros hay por Response, y cuántos tienen Plot en N/A o vacío.

In [3]:
df.group_by("Response").len()

Response,len
str,u32
"""True""",47175
"""False""",28


## Descartemos los registros Response = False

In [36]:
df.filter(pl.col("Response") == "False")

imdb_id,Title,Year,Rated,Released,Runtime,Genre,Director,Writer,Actors,Plot,Language,Country,Awards,Poster,Ratings,Metascore,imdbRating,imdbVotes,imdbID,Type,DVD,BoxOffice,Production,Website,Response,Plot_word_count
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[struct[2]],str,str,str,str,str,str,str,str,str,str,u32
"""tt38018909""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""False""",null
"""tt38077500""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""False""",null
"""tt38063392""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""False""",null
"""tt38095263""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""False""",null
"""tt38121182""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""False""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tt38875911""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""False""",null
"""tt38875979""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""False""",null
"""tt38938222""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""False""",null


In [37]:
df = df.filter(pl.col("Response") == "True")

## Cuántos tienen Plot en N/A o vacío

In [59]:
df.filter(
    pl.col("Plot").is_null() | (pl.col("Plot") == "") | (pl.col("Plot") == "N/A")
).shape[0]

211

# Validar identidad y tipo

In [61]:
df["imdbID"].is_duplicated().sum()

0

In [62]:
df["Type"].value_counts()

Type,count
str,u32
"""movie""",47172
"""episode""",2
"""series""",1


In [76]:
df.filter(pl.col("Type") == "series")

imdb_id,Title,Year,Rated,Released,Runtime,Genre,Director,Writer,Actors,Plot,Language,Country,Awards,Poster,Ratings,Metascore,imdbRating,imdbVotes,imdbID,Type,DVD,BoxOffice,Production,Website,Response,Plot_word_count
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[struct[2]],str,str,str,str,str,str,str,str,str,str,u32
"""tt35076553""","""Inspector Zende""","""2025–""","""N/A""","""05 Sep 2025""","""N/A""","""N/A""","""N/A""","""N/A""","""Manoj Bajpayee, Jim Sarbh, Gir…","""N/A""","""Hindi""","""India""","""N/A""","""https://m.media-amazon.com/ima…",[],"""N/A""","""N/A""","""N/A""","""tt35076553""","""series""",null,null,null,null,"""True""",1


# N/A por campo: porcentaje en cada columna que vas a guardar.

In [84]:
columns = ["Title", "Year", "Rated", "Released", "Genre", "Director", "Writer", "Actors", "Plot", "Language", "Country", "Awards", "Poster", "imdbRating", "imdbVotes", "imdbID", "Runtime"]

In [89]:
df.select([
    (
        pl.col(col).is_null() |
        (pl.col(col) == "") |
        (pl.col(col) == "N/A")
    ).sum().alias(col)
    for col in columns
# ]).transpose(include_header=True, column_names=["Missing"])
])

Title,Year,Rated,Released,Genre,Director,Writer,Actors,Plot,Language,Country,Awards,Poster,imdbRating,imdbVotes,imdbID,Runtime
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,13090,609,146,37,1273,118,211,309,48,16029,83,859,379,0,391


# Validar inconsistencias en columnas antes de convertirlas a tipos numéricos

Año de 4 dígitos

In [75]:
df["imdbID", "Plot", "Year", "Type"].filter(
    ~pl.col("Year").str.contains(r"^\d{4}$")
)

imdbID,Plot,Year,Type
str,str,str,str
"""tt35076553""","""N/A""","""2025–""","""series"""


Duración en minutos

In [54]:
df[["imdbID", "Plot", "Runtime"]].filter(
    ~(pl.col("Runtime") == "N/A") &
    ~pl.col("Runtime").str.contains(r"^\d+ min$")
)

imdbID,Plot,Runtime
str,str,str


Votos numéricos con comas

In [57]:
df[["imdbID", "Plot", "imdbVotes"]].filter(
    ~(pl.col("imdbVotes") == "N/A") &
    ~pl.col("imdbVotes").str.contains(r"^[\d,]+$")
)

imdbID,Plot,imdbVotes
str,str,str


# Validar columnas listas: cuántos valores distintos tienen Genre y Language, y si aparecen separadores raros.

In [50]:
separators = {
    "semicolon": r";",
    "pipe": r"\|",
    "slash": r"/",
    "backslash": r"\\",
    "ampersand": r"&",
}

for col in ["Genre", "Language"]:
    print(f"\n--- Separadores en {col} ---")

    for name, pattern in separators.items():
        count = df.filter(
            pl.col(col).str.contains(pattern).fill_null(False)
        ).height

        if count > 0:
            print(f"{name}: {count} registros")


--- Separadores en Genre ---
slash: 146 registros

--- Separadores en Language ---
slash: 309 registros


In [26]:
for col in ["Genre", "Language"]:
    print(f"\n--- Valores individuales de {col} ---")

    result = (
        df.select(
            pl.col(col).str.split(",").alias("value")
        )
        .explode("value")
        .with_columns(
            pl.col("value").str.strip_chars()
        )
        .filter(
            pl.col("value").is_not_null() &
            (pl.col("value") != "") &
            (pl.col("value") != "N/A")
        )
        .group_by("value")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
    )

    print(result)


--- Valores individuales de Genre ---
shape: (26, 2)
┌───────────┬───────┐
│ value     ┆ count │
│ ---       ┆ ---   │
│ str       ┆ u32   │
╞═══════════╪═══════╡
│ Drama     ┆ 25930 │
│ Comedy    ┆ 15666 │
│ Action    ┆ 8600  │
│ Romance   ┆ 7851  │
│ Crime     ┆ 7703  │
│ …         ┆ …     │
│ Film-Noir ┆ 453   │
│ News      ┆ 31    │
│ Adult     ┆ 17    │
│ Short     ┆ 7     │
│ Talk-Show ┆ 1     │
└───────────┴───────┘

--- Valores individuales de Language ---
shape: (260, 2)
┌──────────┬───────┐
│ value    ┆ count │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ English  ┆ 30665 │
│ French   ┆ 5157  │
│ Spanish  ┆ 3622  │
│ German   ┆ 2765  │
│ Italian  ┆ 2472  │
│ …        ┆ …     │
│ Konkani  ┆ 1     │
│ Kabyle   ┆ 1     │
│ Shuar    ┆ 1     │
│ Inupiaq  ┆ 1     │
│ Rhaetian ┆ 1     │
└──────────┴───────┘


In [65]:
print("Genre == 'None':",df.filter(pl.col("Genre").str.contains(r"\bNone\b")).height)

Genre == 'None': 0


In [64]:
print("Language == 'None':",df.filter(pl.col("Language").str.contains(r"\bNone\b")).height)

Language == 'None': 206


In [82]:
df["imdbID", "Title", "Plot", "Year", "Genre", "Type", "Language"].filter(pl.col("Language").str.contains(r"\bNone\b"))

imdbID,Title,Plot,Year,Genre,Type,Language
str,str,str,str,str,str,str
"""tt0002423""","""Passion""","""The story of Madame DuBarry, t…","""1919""","""Biography, Drama, Romance""","""movie""","""None, German"""
"""tt0004181""","""Judith of Bethulia""","""A fascinating work of high art…","""1914""","""Drama""","""movie""","""None, English"""
"""tt0004465""","""The Perils of Pauline""","""Young Pauline is left a lot of…","""1914""","""Action, Adventure, Drama""","""movie""","""None, English"""
"""tt0002844""","""Fantômas: In the Shadow of the…","""Princess Sonia Danidoff is sta…","""1913""","""Crime, Drama""","""movie""","""None, French"""
"""tt0005060""","""Carmen""","""A group of gypsy smugglers are…","""1915""","""Drama""","""movie""","""None, English"""
…,…,…,…,…,…,…
"""tt5450084""","""Homo Sapiens""","""The images could be taken from…","""2016""","""Documentary""","""movie""","""None"""
"""tt5494830""","""Utopia, an Adults FairyTale""","""Once upon a time, Achilles, an…","""2016""","""Comedy""","""movie""","""None"""
"""tt6848928""","""Junk Head""","""Mankind can no longer reproduc…","""2017""","""Animation, Action, Adventure""","""movie""","""None"""


# Largo del plot: percentiles 50, 90 y 99 en palabras.

Cantidad de palabras de cada plot

In [66]:
def missing(c):
    return pl.col(c).is_null() | pl.col(c).str.strip_chars().is_in(["", "N/A"])

In [ ]:
plots = df.filter(~missing("Plot")).with_columns(
    pl.col("Plot").str.count_matches(r"\S+").alias("words")     # Count words in Plot
)

In [70]:
plots["words"].describe()

statistic,value
str,f64
"""count""",46964.0
"""null_count""",0.0
"""mean""",79.735265
"""std""",62.846458
"""min""",1.0
"""25%""",36.0
"""50%""",65.0
"""75%""",105.0
"""max""",1199.0


In [74]:
plots.select([
    pl.col("words").quantile(0.50).alias("P50"),
    pl.col("words").quantile(0.90).alias("P90"),
    pl.col("words").quantile(0.95).alias("P95")
])

P50,P90,P95
f64,f64,f64
65.0,149.0,195.0


In [68]:
print(plots["words"].min(), plots["words"].quantile(0.5), (plots["words"] > 190).mean())

1 65.0 0.05331743463078102


## Número esperado de películas

In [86]:
df.filter(~missing("Plot") & (pl.col("Type") == "movie")).height

46963